# Huấn luyện và Testing mô hình dự đoán bệnh tim trên Google Colab

Notebook này dùng `heart.csv` của dự án để:
- khám phá và kiểm tra chất lượng dữ liệu;
- huấn luyện 8 thuật toán Machine Learning;
- đánh giá bằng Stratified 5-Fold Cross-Validation và tập test độc lập;
- so sánh Accuracy, Precision, Recall, F1-score và ROC-AUC;
- lưu mô hình tốt nhất cùng toàn bộ mô hình để tải xuống.

> Kết quả chỉ phục vụ mục đích học tập/nghiên cứu, không thay thế chẩn đoán y khoa.

In [ ]:
# Cài thư viện cần thiết trên Google Colab
!pip -q install xgboost joblib seaborn

## 1. Import thư viện và cấu hình

In [ ]:
import random
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TEST_SIZE = 0.20
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

FEATURES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope",
    "ca", "thal"
]
TARGET = "target"

## 2. Đưa file `heart.csv` lên Colab

Chạy cell dưới đây và chọn file `heart.csv` trên máy tính. Nếu file đã nằm trong thư mục làm việc của Colab thì notebook sẽ dùng file đó.

In [ ]:
CSV_PATH = Path("heart.csv")

if not CSV_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_names:
        raise FileNotFoundError("Vui lòng upload file CSV, ví dụ heart.csv")
    CSV_PATH = Path(csv_names[0])

df = pd.read_csv(CSV_PATH)
required_columns = FEATURES + [TARGET]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Thiếu cột bắt buộc: {missing_columns}")

df = df[required_columns].copy()
for column in required_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print(f"Đã đọc: {CSV_PATH}")
print(f"Kích thước ban đầu: {df.shape}")
print(f"Số giá trị thiếu: {int(df.isna().sum().sum())}")
df = df.dropna().reset_index(drop=True)
df[TARGET] = df[TARGET].astype(int)
print(f"Kích thước sau làm sạch: {df.shape}")
display(df.head())
display(df[TARGET].value_counts().rename_axis(TARGET).to_frame("count"))

## 3. Khám phá dữ liệu nhanh (EDA)

In [ ]:
display(df.describe().T)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x=TARGET, ax=axes[0], palette="Set2")
axes[0].set_title("Phân bố nhãn target")
axes[0].set_xlabel("0 = Không bệnh, 1 = Có bệnh")
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Ma trận tương quan")
plt.tight_layout()
plt.show()

## 4. Chia dữ liệu Train/Test và xây dựng 8 mô hình

Tập test được giữ độc lập và chỉ dùng ở bước đánh giá cuối. Các mô hình cần chuẩn hóa được đặt trong `Pipeline`, vì vậy `StandardScaler` chỉ học từ dữ liệu train trong từng fold, tránh data leakage.

In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} mẫu | Test: {X_test.shape[0]} mẫu")
print("Tỷ lệ nhãn train:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Tỷ lệ nhãn test :", y_test.value_counts(normalize=True).round(3).to_dict())

def make_pipeline(model, needs_scaling=False):
    scaler = StandardScaler() if needs_scaling else "passthrough"
    return Pipeline([("scaler", scaler), ("model", model)])

models = {
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), True),
    "Decision Tree": make_pipeline(DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)),
    "K-Nearest Neighbors": make_pipeline(KNeighborsClassifier(n_neighbors=5), True),
    "Support Vector Machine": make_pipeline(SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE), True),
    "Random Forest": make_pipeline(RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
    "Naive Bayes": make_pipeline(GaussianNB(), True),
    "Gradient Boosting": make_pipeline(GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE)),
    "XGBoost": make_pipeline(XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1, tree_method="hist")),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "roc_auc": "roc_auc", "f1": "f1"}

## 5. Huấn luyện và Cross-Validation trên tập train

In [ ]:
cv_rows = []
cv_fold_scores = {}

for name, model in models.items():
    cv_result = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=False, n_jobs=-1)
    cv_fold_scores[name] = cv_result
    cv_rows.append({
        "Model": name,
        "CV Accuracy mean": cv_result["test_accuracy"].mean(),
        "CV Accuracy std": cv_result["test_accuracy"].std(),
        "CV ROC-AUC mean": cv_result["test_roc_auc"].mean(),
        "CV F1 mean": cv_result["test_f1"].mean(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values("CV ROC-AUC mean", ascending=False)
display(cv_results.style.format({"CV Accuracy mean": "{:.3f}", "CV Accuracy std": "{:.3f}", "CV ROC-AUC mean": "{:.3f}", "CV F1 mean": "{:.3f}"}))

fitted_models = {}
test_rows = []
predictions = {}
probabilities = {}

for name, model in models.items():
    fitted_model = clone(model).fit(X_train, y_train)
    y_pred = fitted_model.predict(X_test)
    y_proba = fitted_model.predict_proba(X_test)[:, 1]
    fitted_models[name] = fitted_model
    predictions[name] = y_pred
    probabilities[name] = y_proba
    test_rows.append({
        "Model": name,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred, zero_division=0),
        "Test Recall": recall_score(y_test, y_pred, zero_division=0),
        "Test F1": f1_score(y_test, y_pred, zero_division=0),
        "Test ROC-AUC": roc_auc_score(y_test, y_proba),
    })

test_results = pd.DataFrame(test_rows).sort_values("Test ROC-AUC", ascending=False)
display(test_results.style.format({column: "{:.3f}" for column in test_results.columns if column != "Model"}))

## 6. Testing: Classification Report và Confusion Matrix

In [ ]:
best_name = test_results.iloc[0]["Model"]
print(f"Mô hình tạm chọn theo Test ROC-AUC: {best_name}")

for name in test_results["Model"]:
    print("\n" + "=" * 72)
    print(name)
    print(classification_report(y_test, predictions[name], target_names=["Không bệnh (0)", "Có bệnh (1)"], zero_division=0))

n_models = len(fitted_models)
n_cols = 4
fig, axes = plt.subplots((n_models + n_cols - 1) // n_cols, n_cols, figsize=(16, 9))
axes = np.array(axes).reshape(-1)
for ax, (name, y_pred) in zip(axes, predictions.items()):
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Dự đoán")
    ax.set_ylabel("Thực tế")
for ax in axes[n_models:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Biểu đồ so sánh kết quả testing và ROC Curve

In [ ]:
metric_columns = ["Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC"]
plot_df = test_results.set_index("Model")[metric_columns]
ax = plot_df.plot(kind="barh", figsize=(13, 6), colormap="viridis")
ax.set_xlim(0, 1.05)
ax.set_xlabel("Điểm số")
ax.set_title("So sánh các chỉ số trên tập test")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 7))
for name, y_proba in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.6, label="Ngẫu nhiên")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves trên tập test")
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 8. Phân tích lỗi của mô hình tốt nhất

Đặc biệt kiểm tra False Negative (bệnh thật nhưng mô hình dự đoán không bệnh), vì đây là lỗi cần lưu ý trong bài toán y tế.

In [ ]:
best_pred = predictions[best_name]
best_proba = probabilities[best_name]
test_with_result = X_test.copy()
test_with_result["actual"] = y_test
test_with_result["predicted"] = best_pred
test_with_result["probability_disease"] = best_proba

false_negatives = test_with_result[(test_with_result["actual"] == 1) & (test_with_result["predicted"] == 0)]
print(f"Số False Negative của {best_name}: {len(false_negatives)}")
display(false_negatives)

## 9. Lưu model và kết quả để dùng lại

File bundle chứa pipeline đã fit, danh sách đặc trưng, tên nhãn và kết quả đánh giá. Pipeline đã bao gồm bước chuẩn hóa nếu mô hình cần chuẩn hóa, nên có thể dùng trực tiếp cho dữ liệu bệnh nhân mới.

In [ ]:
ARTIFACT_DIR = Path("heart_model_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

bundle = {
    "best_model_name": best_name,
    "best_model": fitted_models[best_name],
    "all_models": fitted_models,
    "feature_names": FEATURES,
    "target_name": TARGET,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "test_results": test_results,
}

bundle_path = ARTIFACT_DIR / "heart_model_bundle.joblib"
metrics_path = ARTIFACT_DIR / "test_metrics.csv"
test_predictions_path = ARTIFACT_DIR / "test_predictions.csv"

joblib.dump(bundle, bundle_path)
test_results.to_csv(metrics_path, index=False)
test_with_result.to_csv(test_predictions_path, index=False)

print("Đã tạo:")
for path in [bundle_path, metrics_path, test_predictions_path]:
    print(f"- {path}")

## 10. Thử dự đoán một bệnh nhân mới

Thay các giá trị trong `new_patient` bằng dữ liệu thật cần dự đoán.

In [ ]:
new_patient = pd.DataFrame([{
    "age": 55, "sex": 1, "cp": 1, "trestbps": 140,
    "chol": 240, "fbs": 0, "restecg": 1, "thalach": 150,
    "exang": 0, "oldpeak": 1.0, "slope": 1, "ca": 0, "thal": 2
}])[FEATURES]

patient_pred = fitted_models[best_name].predict(new_patient)[0]
patient_prob = fitted_models[best_name].predict_proba(new_patient)[0, 1]
print(f"Mô hình: {best_name}")
print(f"Dự đoán: {patient_pred} (0 = Không bệnh, 1 = Có bệnh)")
print(f"Xác suất lớp Có bệnh: {patient_prob:.2%}")

In [ ]:
# Tải bundle và các file kết quả về máy tính (chạy khi cần)
from google.colab import files
files.download(str(bundle_path))
files.download(str(metrics_path))
files.download(str(test_predictions_path))